# PARC2026 — 76b M3 reverse screening evaluation

75b逆順training PASS後に、逆順の6 checkpoint（3モデル × equal-data/equal-wall）をLIBERO全40 task × 2 seed × 1 trialで評価します。**80 episodes/checkpointのM3選抜用評価**です。最終候補用800 episodes評価は開始しません。


In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_reverse_screening'
PIN = '659cbc51f84a466986dd7698b8d362358f3a24e0'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('76b reverse screening code:', got, flush=True)
env = os.environ.copy()
env['PY_AI_REPO'] = str(REPO)
env['PARC_ROOT'] = str(ROOT)
env['PARC_DRIVE_ROOT'] = '/content/drive/MyDrive/parc2026-cache'
env['PARC_M3_EXECUTE'] = '1'
subprocess.run([
    'python', '-u', str(REPO / 'tools/colab/run_m3_screening_evaluation_order.py'),
    '--order', 'reverse', '--attempt', '1'
], cwd=str(REPO), env=env, check=True)
print('=== 76b COMPLETE ===', flush=True)
print('Reverse screening evaluation complete. Final 800-episode evaluation has NOT started.', flush=True)
